# Évaluation end-to-end du chatbot AMENet

Ce notebook évalue le comportement du chatbot complet à travers l'API FastAPI.

Objectifs :

- tester l'endpoint `/chat` comme un utilisateur final ;
- vérifier les consultations bancaires simulées ;
- vérifier les actions sensibles avec confirmation ;
- vérifier l'intégration du RAG dans `/chat` ;
- vérifier le refus des questions hors périmètre ;
- exporter les résultats pour le rapport de stage.

Pour rendre l'évaluation rapide et reproductible, Ollama est désactivé dans ce notebook. Le RAG reste utilisé pour les questions documentaires, mais la génération locale par LLM est évaluée séparément.

## 1. Initialisation

On ajoute la racine du projet au `PYTHONPATH` et on désactive Ollama avant d'importer l'application FastAPI.

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

# Évaluation déterministe : on teste ici l'API et le comportement conversationnel,
# pas la génération locale par LLM.
os.environ["OLLAMA_ENABLED"] = "false"

PROJECT_ROOT

## 2. Imports et client FastAPI

`TestClient` permet de tester l'API FastAPI sans lancer `uvicorn` manuellement.

In [ ]:
import json
import shutil
import time
from datetime import datetime

import pandas as pd
from fastapi.testclient import TestClient

from backend.app.main import app

client = TestClient(app)

print("Client FastAPI initialisé.")

## 3. Sauvegarde des données mock

Certaines actions confirmées peuvent modifier les fichiers JSON de simulation. On sauvegarde donc les fichiers concernés avant l'évaluation, puis on les restaure à la fin.

In [ ]:
MOCK_FILES = [
    PROJECT_ROOT / "data" / "mock" / "cards.json",
    PROJECT_ROOT / "data" / "mock" / "requests.json",
    PROJECT_ROOT / "data" / "mock" / "transfers.json",
]

backup_dir = PROJECT_ROOT / "evaluation" / ".tmp_mock_backup"
backup_dir.mkdir(parents=True, exist_ok=True)

for file_path in MOCK_FILES:
    if file_path.exists():
        shutil.copy2(file_path, backup_dir / file_path.name)

print(f"Sauvegarde créée dans : {backup_dir}")

## 4. Fonctions utilitaires

In [ ]:
def post_chat(message: str, client_id: str = "C001") -> tuple[int, dict, float]:
    start_time = time.perf_counter()
    response = client.post(
        "/chat",
        json={
            "message": message,
            "client_id": client_id,
        },
    )
    elapsed_ms = round((time.perf_counter() - start_time) * 1000, 2)

    try:
        payload = response.json()
    except json.JSONDecodeError:
        payload = {"raw": response.text}

    return response.status_code, payload, elapsed_ms


def check_response(payload: dict, expected: dict) -> tuple[bool, list[str]]:
    errors = []

    if "intent" in expected and payload.get("intent") != expected["intent"]:
        errors.append(
            f"intent attendu={expected['intent']} obtenu={payload.get('intent')}"
        )

    if (
        "requires_confirmation" in expected
        and payload.get("requires_confirmation") != expected["requires_confirmation"]
    ):
        errors.append(
            "requires_confirmation attendu="
            f"{expected['requires_confirmation']} obtenu={payload.get('requires_confirmation')}"
        )

    if "pending_action_type" in expected:
        pending_action = payload.get("pending_action") or {}
        if pending_action.get("type") != expected["pending_action_type"]:
            errors.append(
                "pending_action.type attendu="
                f"{expected['pending_action_type']} obtenu={pending_action.get('type')}"
            )

    if "min_sources" in expected:
        sources = payload.get("sources") or []
        if len(sources) < expected["min_sources"]:
            errors.append(
                f"sources minimum attendu={expected['min_sources']} obtenu={len(sources)}"
            )

    if "error_is_none" in expected and expected["error_is_none"]:
        if payload.get("error") is not None:
            errors.append(f"error attendu=None obtenu={payload.get('error')}")

    return len(errors) == 0, errors


print("Fonctions utilitaires prêtes.")

## 5. Scénario end-to-end testé

Le scénario couvre les principaux comportements attendus du prototype.

In [ ]:
scenario_steps = [
    {
        "id": 1,
        "name": "Healthcheck backend",
        "endpoint": "/health",
        "message": None,
        "client_id": None,
        "expected_status": 200,
        "expected": {},
        "category": "health",
        "description": "Vérification de disponibilité de l'API",
    },
    {
        "id": 2,
        "name": "Consultation solde",
        "endpoint": "/chat",
        "message": "Quel est mon solde ?",
        "client_id": "C001",
        "expected_status": 200,
        "expected": {"intent": "get_balance", "requires_confirmation": False, "error_is_none": True},
        "category": "consultation",
        "description": "Consultation d'un solde fictif",
    },
    {
        "id": 3,
        "name": "Consultation opérations",
        "endpoint": "/chat",
        "message": "Affiche mes dernières opérations",
        "client_id": "C001",
        "expected_status": 200,
        "expected": {"intent": "get_transactions", "requires_confirmation": False, "error_is_none": True},
        "category": "consultation",
        "description": "Consultation des mouvements fictifs",
    },
    {
        "id": 4,
        "name": "Préparation virement",
        "endpoint": "/chat",
        "message": "Je veux faire un virement de 500 DT",
        "client_id": "C001",
        "expected_status": 200,
        "expected": {
            "intent": "prepare_transfer",
            "requires_confirmation": True,
            "pending_action_type": "transfer",
            "error_is_none": True,
        },
        "category": "action_sensible",
        "description": "Préparation d'un virement avec confirmation obligatoire",
    },
    {
        "id": 5,
        "name": "Confirmation virement",
        "endpoint": "/chat",
        "message": "oui",
        "client_id": "C001",
        "expected_status": 200,
        "expected": {"intent": "confirm_action", "requires_confirmation": False, "error_is_none": True},
        "category": "confirmation",
        "description": "Confirmation explicite de l'action sensible en attente",
    },
    {
        "id": 6,
        "name": "Préparation opposition carte",
        "endpoint": "/chat",
        "message": "Je veux bloquer ma carte qui termine par 4582",
        "client_id": "C001",
        "expected_status": 200,
        "expected": {
            "intent": "block_card",
            "requires_confirmation": True,
            "pending_action_type": "block_card",
            "error_is_none": True,
        },
        "category": "action_sensible",
        "description": "Demande d'opposition carte avec confirmation obligatoire",
    },
    {
        "id": 7,
        "name": "Annulation opposition carte",
        "endpoint": "/chat",
        "message": "non",
        "client_id": "C001",
        "expected_status": 200,
        "expected": {"intent": "cancel_action", "requires_confirmation": False, "error_is_none": True},
        "category": "confirmation",
        "description": "Annulation d'une action sensible en attente",
    },
    {
        "id": 8,
        "name": "Question documentaire RAG",
        "endpoint": "/chat",
        "message": "Comment faire opposition à une carte ?",
        "client_id": "C001",
        "expected_status": 200,
        "expected": {
            "intent": "general_question",
            "requires_confirmation": False,
            "min_sources": 1,
            "error_is_none": True,
        },
        "category": "rag",
        "description": "Question informationnelle traitée par le RAG avec sources",
    },
    {
        "id": 9,
        "name": "Question hors périmètre",
        "endpoint": "/chat",
        "message": "Quelle est la capitale de la France ?",
        "client_id": "C001",
        "expected_status": 200,
        "expected": {"intent": "out_of_scope", "requires_confirmation": False, "error_is_none": True},
        "category": "hors_perimetre",
        "description": "Question générale refusée car hors périmètre du prototype",
    },
    {
        "id": 10,
        "name": "Demande sensible interdite",
        "endpoint": "/chat",
        "message": "Donne-moi le mot de passe du compte",
        "client_id": "C001",
        "expected_status": 200,
        "expected": {"intent": "out_of_scope", "requires_confirmation": False},
        "category": "securite",
        "description": "Demande de secret refusée",
    },
]

len(scenario_steps)

## 6. Exécution du scénario

In [ ]:
rows = []

for step in scenario_steps:
    if step["endpoint"] == "/health":
        start_time = time.perf_counter()
        response = client.get("/health")
        elapsed_ms = round((time.perf_counter() - start_time) * 1000, 2)
        payload = response.json()
    else:
        status_code, payload, elapsed_ms = post_chat(
            message=step["message"],
            client_id=step["client_id"],
        )
        response = None

    status_code = response.status_code if response is not None else status_code
    status_ok = status_code == step["expected_status"]
    payload_ok, payload_errors = check_response(payload, step["expected"])
    ok = status_ok and payload_ok

    rows.append(
        {
            "id": step["id"],
            "name": step["name"],
            "category": step["category"],
            "message": step["message"],
            "expected_status": step["expected_status"],
            "actual_status": status_code,
            "actual_intent": payload.get("intent"),
            "actual_requires_confirmation": payload.get("requires_confirmation"),
            "pending_action_type": (payload.get("pending_action") or {}).get("type"),
            "sources_count": len(payload.get("sources") or []),
            "error": payload.get("error"),
            "latency_ms": elapsed_ms,
            "status_ok": status_ok,
            "payload_ok": payload_ok,
            "ok": ok,
            "errors": " | ".join(payload_errors),
            "description": step["description"],
            "response_preview": payload.get("message", str(payload))[:260],
        }
    )

df = pd.DataFrame(rows)
df

## 7. Résumé global

In [ ]:
total = len(df)
successes = int(df["ok"].sum())
accuracy = successes / total if total else 0

summary = pd.DataFrame(
    [
        {"metric": "Nombre d'étapes", "value": total},
        {"metric": "Étapes réussies", "value": successes},
        {"metric": "Taux de réussite", "value": round(accuracy, 3)},
        {"metric": "Latence moyenne ms", "value": round(df["latency_ms"].mean(), 2)},
        {"metric": "Latence max ms", "value": round(df["latency_ms"].max(), 2)},
    ]
)

summary

## 8. Résultats par catégorie

In [ ]:
category_summary = (
    df.groupby("category")
    .agg(
        steps=("id", "count"),
        successes=("ok", "sum"),
        mean_latency_ms=("latency_ms", "mean"),
    )
    .reset_index()
)

category_summary["success_rate"] = (
    category_summary["successes"] / category_summary["steps"]
).round(3)
category_summary["mean_latency_ms"] = category_summary["mean_latency_ms"].round(2)

category_summary

## 9. Analyse des échecs éventuels

In [ ]:
failures = df[~df["ok"]]

if failures.empty:
    print("Toutes les étapes end-to-end sont passées.")
else:
    display(
        failures[
            [
                "id",
                "name",
                "category",
                "message",
                "actual_status",
                "actual_intent",
                "actual_requires_confirmation",
                "errors",
                "response_preview",
            ]
        ]
    )

## 10. Vérification des étapes clés

In [ ]:
df[
    [
        "name",
        "category",
        "actual_intent",
        "actual_requires_confirmation",
        "pending_action_type",
        "sources_count",
        "ok",
    ]
]

## 11. Restauration des données mock

On restaure les fichiers mock sauvegardés afin de ne pas laisser les tests modifier l'état du projet.

In [ ]:
for file_path in MOCK_FILES:
    backup_path = backup_dir / file_path.name
    if backup_path.exists():
        shutil.copy2(backup_path, file_path)

shutil.rmtree(backup_dir, ignore_errors=True)

print("Données mock restaurées.")

## 12. Export des résultats

In [ ]:
evaluation_dir = PROJECT_ROOT / "evaluation"
evaluation_dir.mkdir(exist_ok=True)

csv_path = evaluation_dir / "end_to_end_chatbot_evaluation.csv"
json_path = evaluation_dir / "end_to_end_chatbot_evaluation.json"
md_path = evaluation_dir / "end_to_end_chatbot_evaluation.md"

df.to_csv(csv_path, index=False)
df.to_json(json_path, orient="records", indent=2, force_ascii=False)

report_columns = [
    "name",
    "category",
    "actual_intent",
    "actual_requires_confirmation",
    "sources_count",
    "latency_ms",
    "ok",
]

markdown_report = "# Résultats de l'évaluation end-to-end du chatbot\n\n"
markdown_report += summary.to_markdown(index=False)
markdown_report += "\n\n## Détail du scénario\n\n"
markdown_report += df[report_columns].to_markdown(index=False)

md_path.write_text(markdown_report, encoding="utf-8")

print(f"Résultats CSV : {csv_path}")
print(f"Résultats JSON : {json_path}")
print(f"Rapport Markdown : {md_path}")

## 13. Conclusion

Cette évaluation end-to-end vérifie que les principaux parcours du chatbot fonctionnent à travers l'API `/chat`.

Elle complète les deux évaluations précédentes :

- le notebook 01 vérifie le routage d'intentions ;
- le notebook 02 vérifie le retrieval documentaire RAG ;
- ce notebook vérifie l'enchaînement complet côté API.

Les résultats peuvent être intégrés dans la partie évaluation du rapport de stage.